# Learning 10: Human in the Loop

**Goal**: Add human approval and input to your workflows

## What You'll Learn
- Interrupting graphs for human input
- Approval workflows
- Resuming after interruption

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Literal
import operator

llm = ChatOpenAI(model="gpt-4o-mini")
print("Setup complete!")

## Why Human in the Loop?

Some actions need human oversight:
- Approving purchases or payments
- Reviewing AI-generated content
- Confirming destructive operations
- Getting additional information

## Example 1: Simple Approval Workflow

In [ ]:
class ApprovalState(TypedDict):
    request: str
    amount: float
    approved: bool
    result: str

def prepare_request(state: ApprovalState) -> dict:
    """Prepare the request for approval."""
    return {"result": f"Request prepared: {state['request']} for ${state['amount']}"}

def process_approved(state: ApprovalState) -> dict:
    """Process after approval."""
    return {"result": f"APPROVED: Processing {state['request']} for ${state['amount']}"}

def process_rejected(state: ApprovalState) -> dict:
    """Handle rejection."""
    return {"result": f"REJECTED: {state['request']} was not approved"}

def check_approval(state: ApprovalState) -> str:
    if state["approved"]:
        return "approved"
    return "rejected"

In [ ]:
# Build graph with interrupt
builder = StateGraph(ApprovalState)

builder.add_node("prepare", prepare_request)
builder.add_node("approved", process_approved)
builder.add_node("rejected", process_rejected)

builder.add_edge(START, "prepare")
builder.add_conditional_edges("prepare", check_approval)
builder.add_edge("approved", END)
builder.add_edge("rejected", END)

memory = MemorySaver()

# Compile with interrupt_before - pauses BEFORE the prepare node
approval_app = builder.compile(
    checkpointer=memory,
    interrupt_before=["prepare"]  # Pause here for human input
)

print("Approval workflow compiled with interrupt!")

In [ ]:
# Start the workflow - it will pause
config = {"configurable": {"thread_id": "approval-1"}}

result = approval_app.invoke(
    {"request": "New laptop", "amount": 1500.0, "approved": False, "result": ""},
    config
)

print("Workflow paused for approval")
print("Current state:", result)

In [ ]:
# Simulate human approval
print("\n--- HUMAN REVIEWS REQUEST ---")
print(f"Request: {result['request']}")
print(f"Amount: ${result['amount']}")
human_decision = True  # Human approves!
print(f"Decision: {'APPROVED' if human_decision else 'REJECTED'}")
print("---")

In [ ]:
# Update state and resume
approval_app.update_state(
    config,
    {"approved": human_decision}
)

# Resume execution
final_result = approval_app.invoke(None, config)
print("Final result:", final_result["result"])

## Example 2: Agent with Tool Approval

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to someone."""
    return f"Email sent to {to} with subject '{subject}'"

@tool
def delete_file(filename: str) -> str:
    """Delete a file from the system."""
    return f"File {filename} deleted"

tools = [send_email, delete_file]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

def agent(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "end"

tool_node = ToolNode(tools)

In [ ]:
# Build agent with interrupt before tools
builder = StateGraph(AgentState)

builder.add_node("agent", agent)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")

memory = MemorySaver()

# Interrupt BEFORE executing tools
agent_app = builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"]  # Human must approve tool execution
)

print("Agent with tool approval compiled!")

In [ ]:
# Start a request that will trigger a tool
config = {"configurable": {"thread_id": "agent-1"}}

result = agent_app.invoke(
    {"messages": [HumanMessage(content="Send an email to bob@example.com about the meeting tomorrow")]},
    config
)

print("Agent wants to use a tool. Paused for approval.")
last_msg = result["messages"][-1]
if hasattr(last_msg, "tool_calls"):
    print("\nPending tool calls:")
    for tc in last_msg.tool_calls:
        print(f"  - {tc['name']}: {tc['args']}")

In [ ]:
# Human approves - resume execution
print("\n--- HUMAN APPROVES TOOL EXECUTION ---\n")

final_result = agent_app.invoke(None, config)
print("Final response:", final_result["messages"][-1].content)

## Example 3: Getting Human Input Mid-Workflow

In [ ]:
class FormState(TypedDict):
    name: str
    email: str
    step: str
    complete: bool

def ask_name(state: FormState) -> dict:
    return {"step": "name"}

def ask_email(state: FormState) -> dict:
    return {"step": "email"}

def finalize(state: FormState) -> dict:
    return {"complete": True, "step": "done"}

def route_form(state: FormState) -> str:
    if not state.get("name"):
        return "ask_name"
    elif not state.get("email"):
        return "ask_email"
    else:
        return "finalize"

In [ ]:
builder = StateGraph(FormState)

builder.add_node("ask_name", ask_name)
builder.add_node("ask_email", ask_email)
builder.add_node("finalize", finalize)

builder.add_conditional_edges(START, route_form)
builder.add_edge("ask_name", END)  # Pause after asking
builder.add_edge("ask_email", END)  # Pause after asking
builder.add_edge("finalize", END)

memory = MemorySaver()
form_app = builder.compile(checkpointer=memory)

print("Multi-step form compiled!")

In [ ]:
config = {"configurable": {"thread_id": "form-1"}}

# Step 1: Start form
result = form_app.invoke(
    {"name": "", "email": "", "step": "", "complete": False},
    config
)
print(f"Step: {result['step']}")
print("Please enter your name...")

In [ ]:
# Step 2: Provide name, continue
form_app.update_state(config, {"name": "Alice"})
result = form_app.invoke(None, config)
print(f"Step: {result['step']}")
print("Please enter your email...")

In [ ]:
# Step 3: Provide email, complete
form_app.update_state(config, {"email": "alice@example.com"})
result = form_app.invoke(None, config)
print(f"Step: {result['step']}")
print(f"Complete: {result['complete']}")
print(f"\nForm submitted! Name: {result['name']}, Email: {result['email']}")

## Key Takeaways

1. `interrupt_before` pauses execution before specified nodes
2. `interrupt_after` pauses after specified nodes
3. `update_state()` modifies state while paused
4. Resume with `invoke(None, config)`
5. Requires a checkpointer for state persistence

## Congratulations!

You've completed all 10 learning notebooks! You now understand:
- LangChain fundamentals (LLM, prompts, chains, tools)
- LangGraph concepts (state, nodes, edges, loops)
- Advanced patterns (agents, memory, human-in-the-loop)